# Simple timecourse

In this example, a simple timecourse is defined for a simple ODE. Simulation and optimization is demonstrated.

The ODE is $\frac{dx}{dt} = p  q$, with $p$ an estimated parameter, and $q$ a timecourse parameter.

NB: in code, symbols are suffixed with `_` to avoid conflicts with packages.

In [1]:
from itertools import chain
from pathlib import Path

import amici
import numpy as np
import petab
import petab_timecourse

from simple_timecourse_helpers import get_analytical_x_, get_analytical_sx_


petab_path = Path('input') / 'simple_timecourse'
experiment_id = 'experiment1'
true_p_ = 1

In [2]:
petab_problem = petab.Problem.from_yaml(str(petab_path / 'petab.yaml'))

The parameter timecourse is represented graphically here. At the indicated timepoints, the value of $q$ (`q_`) changes.
<img src="input/simple_timecourse/timecourse.png" width="400">

In [3]:
petab.lint.lint_problem(petab_problem)

Visualization table not available. Skipping.


False

In [4]:
experiment = petab.Experiment.from_df(petab_problem.experiment_df, experiment_id=experiment_id)

In [5]:
import pandas as pd
pd.DataFrame(data={"s": [None]}).values.item()

In [6]:
[period.condition_id for period in experiment.periods]

['q_positive', 'q_zero', 'q_negative', 'q_zero', 'q_positive']

In [ ]:
from petab_timecourse.simulator import AmiciSimulator
simulator = AmiciSimulator(petab_problem=petab_problem, experiment_id=experiment_id)
simulator.amici_solver.setSensitivityOrder(1)

> /home/dilan/Documents/future_annex/optimal_control/packages/petab_timecourse/petab_timecourse/amici.py(372)precreate_parameter_mapping_periods()
    371 
--> 372     for petab_problem in petab_problems:
    373         # Create dummy measurement df, for timecourse periods



ipdb>  prelim_parameter_mapping


[[({'p_': 'p_', 'q_': 1.0}, {'p_': 'lin', 'q_': 'lin'}), ({'p_': 'p_', 'q_': 0.0}, {'p_': 'lin', 'q_': 'lin'}), ({'p_': 'p_', 'q_': -1.0}, {'p_': 'lin', 'q_': 'lin'}), ({'p_': 'p_', 'q_': 0.0}, {'p_': 'lin', 'q_': 'lin'}), ({'p_': 'p_', 'q_': 1.0}, {'p_': 'lin', 'q_': 'lin'})]]


In [ ]:
"""

from petab_timecourse.simulator import AmiciSimulator
simulator = AmiciSimulator(petab_problem=petab_problem, timecourse_id='timecourse1')
simulator.amici_solver.setSensitivityOrder(1)

results = simulator.simulate(problem_parameters_periods=[{'p_': v} for v in p_values])

x_ = collect_x(results)
sx_ = collect_sx(results)
T = collect_t(results)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10,10))
ax.plot(T, x_, lw=5, label='Simulated state')
ax.legend()
ax.set_title('State trajectory');
"""

In [ ]:
p_values = [0.5, 2.0, 1.0, 1.5, 2.0]

results = simulator.simulate(problem_parameters_periods=[{'p_': v} for v in p_values])

In [ ]:
from petab_timecourse.amici import collect_x, collect_sx, collect_t


x_ = collect_x(results)
sx_ = collect_sx(results)
T = collect_t(results)

analytical_x_  = [np.round(get_analytical_x_(t, timecourse=timecourse, p_=true_p_), 5)  for t in T]
analytical_sx_ = [np.round(get_analytical_sx_(t, timecourse=timecourse), 5) for t in T]

# The state (x_) trajectory is correct.
#assert np.isclose(x_, analytical_x_).all()
# The state (x_) forward sensitivity w.r.t. the parameter (p_) is correct.
#assert np.isclose(sx_, analytical_sx_).all()

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10,10))
ax.plot(T, x_, lw=5, label='Simulated state')
ax.plot(T, analytical_x_, linestyle=':', lw=10, label='Analytical state')
ax.legend()
ax.set_title('State trajectory');

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))
ax.plot(T, x_, lw=5, label='Simulated state sensitivity')
ax.plot(T, analytical_x_, linestyle=':', lw=10, label='Analytical state sensitivity')
ax.legend()
ax.set_title('State sensitivity trajectory');